In [1]:
import pickle
import numpy as np
import pandas as pd
import os
from typing import Dict, List
import random

In [2]:
def make_folds_greedy(patient_measurements: dict, k=5, seed=42):
    """
    patient_measurements : dict {patient_id: n_meas}
    Devuelve una lista de k listas (IDs de pacientes por fold-test),
    con los totales de mediciones lo más iguales posible.
    """
    rng = random.Random(seed)

    # Ordena por nº de mediciones (desc) y baraja un poco para romper empates
    items = list(patient_measurements.items())
    rng.shuffle(items)
    items.sort(key=lambda x: x[1], reverse=True)

    folds = [[] for _ in range(k)]
    fold_load = [0] * k                       # mediciones acumuladas

    for pid, n in items:
        idx = fold_load.index(min(fold_load)) # fold más “ligero”
        folds[idx].append(pid)
        fold_load[idx] += n                   # actualiza su peso

    return folds


In [3]:
Fold = Dict[str, List[str]]   # {'train': [...], 'val': [...], 'test': [...]}

def make_folds_greedy_splits(   patient_measurements: Dict[str, int],   # ← IDs como str
                                k: int = 5,
                                val_frac: float = 0.10,
                                seed: int = 42) -> List[Fold]:
    """
    Reparte pacientes (clave str) y su nº de mediciones (valor int)
    en k folds balanceando la carga total.
    Devuelve: list[ {'train': [...], 'val': [...], 'test': [...]} ]
    """
    rng = random.Random(seed)

    # 1) asignar pacientes grandes primero al fold-test más ligero
    items = list(patient_measurements.items())   # [(pid, n_meas), ...]
    rng.shuffle(items)
    items.sort(key=lambda x: x[1], reverse=True)

    test_folds: List[List[str]] = [[] for _ in range(k)]
    fold_load  = [0] * k                         # suma de mediciones por fold

    for pid, n in items:
        idx = fold_load.index(min(fold_load))
        test_folds[idx].append(pid)
        fold_load[idx] += n

    # 2) dentro de cada fold-test, crear train y val a partir del resto
    all_ids = list(patient_measurements.keys())
    folds: List[Fold] = []

    for i in range(k):
        test_ids = set(test_folds[i])
        remaining = [pid for pid in all_ids if pid not in test_ids]

        rng.shuffle(remaining)
        cut = int(val_frac * len(remaining))     # 20 % para validación
        val_ids   = remaining[:cut]
        train_ids = remaining[cut:]

        folds.append({
            'train': train_ids,
            'val'  : val_ids,
            'test' : list(test_ids)
        })

    return folds


In [4]:
from collections import Counter
from typing import Dict, List

# Type alias for clarity: each fold is a dict with three lists of patient IDs (str)
Fold = Dict[str, List[str]]      # {'train': [...], 'val': [...], 'test': [...]}

def validate_folds(
        folds: List[Fold],
        patient_measurements: Dict[str, int],
        tolerance_pct: float = 10.0) -> None:
    """
    Validate a list of patient-wise cross-validation folds.

    Parameters
    ----------
    folds : list[Fold]
        Output of `make_folds_greedy_splits` or similar.  Each element is a
        dictionary with keys ``'train'``, ``'val'``, ``'test'`` mapping to lists
        of patient IDs **(strings)**.

    patient_measurements : dict[str, int]
        Mapping ``{patient_id: number_of_measurements}``.  Used to check load
        balancing (Rule R4).

    tolerance_pct : float, default 10.0
        Maximum allowed percentage deviation from the mean test load before
        a warning is raised (Rule R4).

    Raises
    ------
    AssertionError
        If any of the mandatory rules (R1–R3) is violated.

    Side effects
    ------------
    * Prints a success message for each rule that passes.
    * Prints the test-set load per fold and the percentage deviation from the
      mean (always), and a warning if any fold exceeds *tolerance_pct*.

    Notes
    -----
    *R4* (load balance) is **informative**; it does **not** raise an error.  
    Adjust *tolerance_pct* as appropriate for your project.
    """

    all_patients = set(patient_measurements)

    # ------------------------------------------------------------------ R1/R3
    for idx, fold in enumerate(folds):
        tr, va, te = map(set, (fold['train'], fold['val'], fold['test']))

        # R1 – exclusivity within the same fold
        assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), (
            f"[R1] Overlap detected in fold {idx}"
        )

        # R3 – union covers the entire cohort
        assert len(tr | va | te) == len(all_patients), (
            f"[R3] Missing patients in fold {idx}"
        )

    print("✓ R1 and R3 passed for every fold.")

    # ------------------------------------------------------------------ R2
    test_counter = Counter(pid for fold in folds for pid in fold['test'])
    duplicates = [pid for pid, cnt in test_counter.items() if cnt != 1]

    assert not duplicates, (
        f"[R2] Patients appearing in test 0 or >1 times: {duplicates}"
    )
    print("✓ R2 passed – each patient appears in *test* exactly once.")

    # ------------------------------------------------------------------ R4 (informative)
    def test_load(ids: List[str]) -> int:
        """Total number of measurements for this group of patients."""
        return sum(patient_measurements[pid] for pid in ids)

    loads = [test_load(f['test']) for f in folds]
    mean_load = sum(loads) / len(loads)
    pct_dev = [100 * (l - mean_load) / mean_load for l in loads]

    print("\nTest-set load per fold (number of measurements):", loads)
    print("Deviation from mean (%)                      :", [round(p, 2) for p in pct_dev])

    if any(abs(p) > tolerance_pct for p in pct_dev):
        print(f"⚠️  [R4] Warning: at least one fold deviates by more than {tolerance_pct}% "
              "from the mean test load.")
    else:
        print(f"✓ R4 passed – test loads within ±{tolerance_pct}% of the mean.")

    print("\nAll mandatory validation checks completed successfully.\n")


In [5]:
windows_path = 'c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/windows/no_split_2025/'
horizon = 2

print(f"Reading test files from {windows_path} for horizon {horizon}...")
df_windows = pd.read_parquet(f"{windows_path}windows_horizon_{horizon}.parquet")
unique_patient_ids = df_windows['patient_id'].unique()
num_patients = len(unique_patient_ids)
df_windows

Reading test files from c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/windows/no_split_2025/ for horizon 2...


,x0,x1,x2,x3,x4,x5,x6,x7,y,patient_id
0,99,92,86,85,85,87,88,93,134,LIB193263
1,92,86,85,85,87,88,93,106,165,LIB193263
2,86,85,85,87,88,93,106,134,187,LIB193263
3,85,85,87,88,93,106,134,165,206,LIB193263
4,85,87,88,93,106,134,165,187,223,LIB193263
...,...,...,...,...,...,...,...,...,...,...
20037336,187,208,206,191,170,155,148,145,143,LIB194166
20037337,208,206,191,170,155,148,145,141,169,LIB194166
20037338,206,191,170,155,148,145,141,143,206,LIB194166
20037339,191,170,155,148,145,141,143,169,215,LIB194166


In [6]:
patient_number_windows_dict = (
    df_windows
    .groupby("patient_id")   # group all rows that share the same ID
    .size()                  # count rows per group
    .astype(int)             # ensure Python int, not numpy int64
    .to_dict()               # ==> { 'ID_1': n1, 'ID_2': n2, ... }
)
patient_number_windows_dict

{'LIB193263': 53722,
 'LIB193264': 23976,
 'LIB193265': 40920,
 'LIB193266': 40716,
 'LIB193267': 47972,
 'LIB193268': 35901,
 'LIB193269': 65579,
 'LIB193272': 3136,
 'LIB193273': 14475,
 'LIB193274': 10854,
 'LIB193277': 104703,
 'LIB193278': 3165,
 'LIB193279': 3375,
 'LIB193280': 50421,
 'LIB193281': 3374,
 'LIB193284': 9556,
 'LIB193302': 24678,
 'LIB193303': 4934,
 'LIB193304': 54102,
 'LIB193307': 97527,
 'LIB193308': 15182,
 'LIB193309': 64058,
 'LIB193310': 66722,
 'LIB193311': 22784,
 'LIB193312': 14594,
 'LIB193313': 28968,
 'LIB193314': 55582,
 'LIB193315': 27936,
 'LIB193317': 14155,
 'LIB193318': 16376,
 'LIB193319': 2420,
 'LIB193320': 34699,
 'LIB193324': 23483,
 'LIB193325': 9874,
 'LIB193326': 8956,
 'LIB193328': 3177,
 'LIB193330': 10260,
 'LIB193332': 48985,
 'LIB193333': 17610,
 'LIB193334': 23263,
 'LIB193335': 19093,
 'LIB193337': 4052,
 'LIB193338': 7495,
 'LIB193340': 26881,
 'LIB193341': 51044,
 'LIB193342': 56918,
 'LIB193343': 5285,
 'LIB193344': 9187,
 'LIB

In [7]:
folds_split = make_folds_greedy_splits(patient_number_windows_dict, k=5, val_frac=0.2, seed=42)


In [8]:
for i, fold in enumerate(folds_split):
    print(f"\n---- Fold {i + 1}: -----")
    print("-- Totals --")
    print(f"  Train: {len(fold['train'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['train'])}")
    print(f"  Val:   {len(fold['val'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['val'])}") 
    print(f"  Test:  {len(fold['test'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['test'])}")
    
    print("\n -- Percentages --")
    print(f"  Train: {(len(fold['train'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['train']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Val:   {(len(fold['val'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['val']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    print(f"  Test:  {(len(fold['test'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['test']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")


---- Fold 1: -----
-- Totals --
  Train: 443 patients. Total windows: 12425990
  Val:   110 patients. Total windows: 3604264
  Test:  138 patients. Total windows: 4007087

 -- Percentages --
  Train: 64.11 percentage of patients. Percentage of windows: 62.01%
  Val:   15.92 percentage of patients. Percentage of windows: 17.99%
  Test:  19.97 percentage of patients. Percentage of windows: 20.00%

---- Fold 2: -----
-- Totals --
  Train: 443 patients. Total windows: 12803378
  Val:   110 patients. Total windows: 3227054
  Test:  138 patients. Total windows: 4006909

 -- Percentages --
  Train: 64.11 percentage of patients. Percentage of windows: 63.90%
  Val:   15.92 percentage of patients. Percentage of windows: 16.11%
  Test:  19.97 percentage of patients. Percentage of windows: 20.00%

---- Fold 3: -----
-- Totals --
  Train: 442 patients. Total windows: 13368131
  Val:   110 patients. Total windows: 2659919
  Test:  139 patients. Total windows: 4009291

 -- Percentages --
  Train: 6

In [9]:
from collections import Counter
from typing import List, Dict

def validate_folds(folds: List[Dict[str, list]],
                   patient_measurements: Dict[str, int]) -> None:
    """
    folds : salida de make_folds_greedy_splits()
    patient_measurements : {patient_id: n_meas}
    Lanza AssertionError si encuentra alguna infracción de R1-R3.
    Imprime resumen de balance (R4).
    """
    all_ids = set(patient_measurements)

    # --- R1, R3 por fold ---
    for i, fold in enumerate(folds):
        tr, va, te = map(set, (fold['train'], fold['val'], fold['test']))

        assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), \
            f"Solapamiento en fold {i}"

        assert len(tr | va | te) == len(all_ids), \
            f"Faltan pacientes en fold {i}"

    # --- R2 global ---
    test_counter = Counter(pid for fold in folds for pid in fold['test'])
    duplicates = [pid for pid, c in test_counter.items() if c != 1]
    assert not duplicates, f"Pacientes repetidos o ausentes en test: {duplicates}"

    # --- R4 balance opcional ---
    def fold_load(ids):        # nº de mediciones de este grupo
        return sum(patient_measurements[pid] for pid in ids)

    test_loads = [fold_load(f['test']) for f in folds]
    print("Carga (mediciones) por fold-test:", test_loads)
    print("Desviación % respecto a la media:",
          [round(100*(l - sum(test_loads)/len(test_loads)) / (sum(test_loads)/len(test_loads)),2)
           for l in test_loads])


In [10]:
validate_folds(folds_split, patient_number_windows_dict)

Carga (mediciones) por fold-test: [4007087, 4006909, 4009291, 4007033, 4007021]
Desviación % respecto a la media: [-0.01, -0.01, 0.05, -0.01, -0.01]


In [11]:
import os
from typing import List, Dict
import pandas as pd

Fold = Dict[str, List[str]]     # {'train': [...], 'val': [...], 'test': [...]}

def save_folds_as_parquet(df_windows: pd.DataFrame, folds: List[Fold], out_dir: str) -> None:
    """
    Save every split of every fold into its own Parquet file.

    Parameters
    ----------
    df_windows : pd.DataFrame
        Full set of windows. Must include column 'patient_id'.

    folds : list[Fold]
        Output of make_folds_greedy_splits or equivalent.

    out_dir : str
        Root directory where `fold_0`, `fold_1`, ... will be created.

    Notes
    -----
    Folder layout created::

        out_dir/
            fold_0/
                train.parquet
                val.parquet
                test.parquet
            fold_1/
                ...
    """
    os.makedirs(out_dir, exist_ok=True)

    for i, fold in enumerate(folds):
        fold_dir = os.path.join(out_dir, f"fold_{i}")
        os.makedirs(fold_dir, exist_ok=True)

        for split in ("train", "val", "test"):
            ids = set(fold[split])          # cast to set for fast lookup
            df_split = df_windows[df_windows["patient_id"].isin(ids)]

            file_path = os.path.join(fold_dir, f"{split}.parquet")
            df_split.to_parquet(file_path, index=False)

        print(f"✓ Fold {i} saved to {fold_dir}")

    print(f"\nAll {len(folds)} folds written under: {out_dir}\n")


In [57]:
save_folds_as_parquet(df_windows, folds_split, "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/")

✓ Fold 0 saved to c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/fold_0
✓ Fold 1 saved to c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/fold_1
✓ Fold 2 saved to c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/fold_2
✓ Fold 3 saved to c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/fold_3
✓ Fold 4 saved to c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/fold_4

All 5 folds written under: c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/


In [12]:
import pandas as pd
from typing import List, Dict

Fold = Dict[str, List[str]]    # {'train': [...], 'val': [...], 'test': [...]}

def tag_folds_and_save(
        df_windows: pd.DataFrame,
        folds: List[Fold],
        output_path: str,
        column_prefix: str = "fold") -> None:
    """
    Add one column per fold (fold0 … fold{k-1}) indicating whether each window
    belongs to 'train', 'val' or 'test' in that fold, then write a single
    Parquet file.

    Parameters
    ----------
    df_windows : pd.DataFrame
        Original window table (must include column 'patient_id').

    folds : list[Fold]
        Output from make_folds_greedy_splits (patient IDs per split).

    output_path : str
        Destination filename, e.g. 'windows_with_folds.parquet'.

    column_prefix : str, default "fold"
        Prefix for the new columns: fold0, fold1, …

    parquet_kwargs : dict, optional
        Extra arguments passed to DataFrame.to_parquet.
    """    
    df_tagged = df_windows.copy()

    for idx, fold in enumerate(folds):
        # Build a mapping patient_id -> split label for this fold
        label_map = {pid: "train" for pid in fold["train"]}
        label_map.update({pid: "val"  for pid in fold["val"]})
        label_map.update({pid: "test" for pid in fold["test"]})

        col_name = f"{column_prefix}{idx}"
        df_tagged[col_name] = df_tagged["patient_id"].map(label_map).astype("category")

    os.makedirs(output_path, exist_ok=True)
    
    # df_tagged.to_parquet(output_path, index=False)
    print(f"✓ File written: {output_path}  "
          f"({len(df_windows):,} rows, {len(folds)} fold-columns)")
    
    return df_tagged

In [13]:
df_all = tag_folds_and_save(df_windows, folds_split, "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/all_folds/")

✓ File written: c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/all_folds/  (20,037,341 rows, 5 fold-columns)


In [21]:
df_all.to_parquet("c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/all_folds/windows_with_folds.parquet", index=False)

In [14]:
df_all.head()

,x0,x1,x2,x3,x4,x5,x6,x7,y,patient_id,fold0,fold1,fold2,fold3,fold4
0,99,92,86,85,85,87,88,93,134,LIB193263,test,val,train,train,val
1,92,86,85,85,87,88,93,106,165,LIB193263,test,val,train,train,val
2,86,85,85,87,88,93,106,134,187,LIB193263,test,val,train,train,val
3,85,85,87,88,93,106,134,165,206,LIB193263,test,val,train,train,val
4,85,87,88,93,106,134,165,187,223,LIB193263,test,val,train,train,val


In [19]:
df_fold_0_train = df_all[df_all['fold1'] == 'train']
df_fold_0_val = df_all[df_all['fold1'] == 'val']
df_fold_0_test = df_all[df_all['fold1'] == 'test']

In [20]:
df_fold_0_val

,x0,x1,x2,x3,x4,x5,x6,x7,y,patient_id,fold0,fold1,fold2,fold3,fold4
0,99,92,86,85,85,87,88,93,134,LIB193263,test,val,train,train,val
1,92,86,85,85,87,88,93,106,165,LIB193263,test,val,train,train,val
2,86,85,85,87,88,93,106,134,187,LIB193263,test,val,train,train,val
3,85,85,87,88,93,106,134,165,206,LIB193263,test,val,train,train,val
4,85,87,88,93,106,134,165,187,223,LIB193263,test,val,train,train,val
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20010576,107,108,106,99,87,78,67,57,55,LIB194163,test,val,train,train,train
20010577,108,106,99,87,78,67,57,56,59,LIB194163,test,val,train,train,train
20010578,106,99,87,78,67,57,56,55,68,LIB194163,test,val,train,train,train
20010579,99,87,78,67,57,56,55,59,72,LIB194163,test,val,train,train,train
